# Feature Engineering — Report Usage Marts

## Purpose

This notebook builds the complete feature layer from processed semantic-model tables in
`data/processed/`. It produces a **continuous daily report-usage series** and a
**separate diagnostic context mart**.

Every major computation delegates to a reusable function in `src/`.
No business logic is implemented here — cells call, display, and verify.

### Output grain and purpose

| Mart | Grain | Purpose |
|---|---|---|
| `mart_report_daily_series` | report_id × date | SARIMA input — `daily_views` only |
| `mart_report_daily_context` | report_id × date | Diagnostic context for analytics, Streamlit, GenAI |

### Current SARIMA model is univariate

The SARIMA model reads **only** `date`, `report_id`, and `daily_views` from
`mart_report_daily_series`. Engagement and performance columns in
`mart_report_daily_context` are **diagnostic** — they describe report health
but are **not** passed to the model.

### Leakage-safe rolling features

Rolling features apply `shift(1)` before each window so the window at day *t*
covers [*t*-1, *t*-2, …] only. This prevents `daily_views` from appearing
in its own predictor. The first row per report is `NaN` for all rolling columns
because no prior-day data exists.

## 1. Imports and Project Setup

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Feature engineering — daily-grain builders
from src.features.report_features import (
    build_report_daily_series,
    build_report_daily_adoption,
    add_time_series_usage_features,
)
from src.features.engagement_features import build_user_engagement_features
from src.features.performance_features import build_report_performance_features
from src.features.build_forecast_features import build_report_daily_context
from src.features.validate_series import validate_report_daily_series, SeriesValidationError
from src.features.feature_registry import (
    get_feature_registry,
    get_predictor_features,
    get_diagnostic_features,
)

# Analytics — report-level aggregation (usage_change_28d_pct, usage_trend_12w_slope)
from src.analytics.report_features import build_report_features

# ── Paths — all relative to project root ──────────────────────────────────────
DATA_DIR = PROJECT_ROOT / "data" / "processed"

FACT_VIEWS_PATH  = DATA_DIR / "fact_report_views.csv"
FACT_PAGES_PATH  = DATA_DIR / "fact_page_views.csv"
FACT_LOADS_PATH  = DATA_DIR / "fact_report_loads.csv"
DIM_REPORT_PATH  = DATA_DIR / "dim_report.csv"
DIM_DATE_PATH    = DATA_DIR / "dim_date.csv"

SERIES_OUT      = DATA_DIR / "mart_report_daily_series.csv"
ADOPTION_OUT    = DATA_DIR / "mart_report_daily_adoption.csv"
TS_OUT          = DATA_DIR / "mart_report_daily_adoption_ts_features.csv"
ENGAGEMENT_OUT  = DATA_DIR / "mart_user_engagement.csv"
PERFORMANCE_OUT = DATA_DIR / "mart_report_performance.csv"
CONTEXT_OUT     = DATA_DIR / "mart_report_daily_context.csv"

print("Project root:", PROJECT_ROOT)

Project root: /Users/masegomodibane/Documents/GitHub/Data Science Projects /Forecasting Report Usage/GitHub Final Version/report-usage-forecasting


## 2. Load Validated Semantic Tables

Five source tables are required. `dim_date` is optional — it resolves integer
`date_key` columns to parsed dates; used by `build_report_features` when the
adoption mart still carries `date_key` rather than a parsed `date`.

All files live in `data/processed/` and were produced by the upstream
semantic-model build (`notebooks/03_data_validation.ipynb`).

In [2]:
for _p in [FACT_VIEWS_PATH, FACT_PAGES_PATH, FACT_LOADS_PATH, DIM_REPORT_PATH]:
    if not _p.exists():
        raise FileNotFoundError(f"Required source file not found: {_p}")

fact_report_views = pd.read_csv(FACT_VIEWS_PATH)
fact_page_views   = pd.read_csv(FACT_PAGES_PATH)
fact_report_loads = pd.read_csv(FACT_LOADS_PATH)
dim_report        = pd.read_csv(DIM_REPORT_PATH)
dim_date          = pd.read_csv(DIM_DATE_PATH) if DIM_DATE_PATH.exists() else pd.DataFrame()

_sources = [
    ("fact_report_views", fact_report_views, "date_key"),
    ("fact_page_views",   fact_page_views,   "date_key"),
    ("fact_report_loads", fact_report_loads, "date_key"),
    ("dim_report",        dim_report,        "launch_date"),
]
print(f"{'Table':<25}  {'Rows':>8}  {'Cols':>5}  Range")
print("-" * 65)
for name, df, key_col in _sources:
    rng = (
        f"{df[key_col].min()} – {df[key_col].max()}"
        if key_col in df.columns else "(absent)"
    )
    print(f"{name:<25}  {len(df):>8,}  {df.shape[1]:>5}  {key_col}: {rng}")

Table                          Rows   Cols  Range
-----------------------------------------------------------------
fact_report_views           135,430      6  date_key: 20250101 – 20260331
fact_page_views             270,635      7  date_key: 20250101 – 20260331
fact_report_loads           135,430      7  date_key: 20250101 – 20260331
dim_report                       30      8  launch_date: 2025-01-01 – 2025-07-07


In [3]:
# dim_report: confirm active-period columns are present
print("dim_report columns:", dim_report.columns.tolist())
display(
    dim_report[["report_id", "archetype", "launch_date", "retire_date"]]
    .sort_values("report_id")
    .head(8)
)

dim_report columns: ['report_id', 'report_name', 'workspace_id', 'report_type', 'is_usage_metrics_report', 'archetype', 'launch_date', 'retire_date']


,report_id,archetype,launch_date,retire_date
0,R_001,stable_weekly,2025-01-01,NaN
1,R_002,stable_weekly,2025-01-01,NaN
2,R_003,stable_weekly,2025-01-01,NaN
3,R_004,upward_adoption,2025-01-01,NaN
4,R_005,upward_adoption,2025-01-01,NaN
5,R_006,upward_adoption,2025-01-01,NaN
6,R_007,gradual_decline,2025-01-01,NaN
7,R_008,gradual_decline,2025-01-01,NaN


## 3. Build the Daily Report Series

`build_report_daily_series` produces **one row per (report_id, date)** covering
every calendar day inside each report's active window.

**Why active no-view dates become zero:** SARIMA requires a contiguous daily
series. If a day is absent from `fact_report_views`, the model would either
skip it (wrong periodicity) or impute silently. Zero-filling makes the
assumption explicit and auditable — the indicator columns distinguish
observed activity from structural absence:

| Column | Meaning |
|---|---|
| `is_observed_day` | At least one event existed for this report-date |
| `is_imputed_zero` | Report was active but had no events; `daily_views` set to 0 |

Active-period anchor from `dim_report.launch_date` / `dim_report.retire_date`:
- Retired reports end at their recorded retire date (not the dataset end).
- Late-launch reports start at their recorded launch date.
- Live reports are active through the dataset-wide maximum observed date.

In [4]:
mart_report_daily_series = build_report_daily_series(
    fact_report_views=fact_report_views,
    report_active_periods=dim_report,
    date_col="date_key",
    report_col="report_id",
    views_col="view_count",
    active_start_col="launch_date",
    active_end_col="retire_date",
)

print(f"Shape:             {mart_report_daily_series.shape}")
print(f"Active-period src: {mart_report_daily_series.attrs['active_period_source']}")
print(f"Observed days:     {mart_report_daily_series['is_observed_day'].sum():,}")
print(f"Imputed zeros:     {mart_report_daily_series['is_imputed_zero'].sum():,}")
print(f"Date range:        {mart_report_daily_series['date'].min().date()} – "
      f"{mart_report_daily_series['date'].max().date()}")
display(mart_report_daily_series.head(4))

Shape:             (11847, 5)
Active-period src: report_active_periods
Observed days:     9,518
Imputed zeros:     2,329
Date range:        2025-01-01 – 2026-03-31


,report_id,date,daily_views,is_observed_day,is_imputed_zero
0,R_001,2025-01-01,39,True,False
1,R_001,2025-01-02,43,True,False
2,R_001,2025-01-03,35,True,False
3,R_001,2025-01-04,10,True,False


In [5]:
# Show a retired report whose last day is an imputed zero
_retired_ids = dim_report[dim_report["retire_date"].notna()]["report_id"]
_rid = _retired_ids.iloc[0]
_r = mart_report_daily_series[mart_report_daily_series["report_id"] == _rid]
print(f"Report {_rid}:  {len(_r)} rows  |  "
      f"{_r['date'].min().date()} → {_r['date'].max().date()}")
print(f"Imputed zero days: {_r['is_imputed_zero'].sum()}")
print("\nLast 4 rows (boundary and any imputed zeros):")
display(_r.tail(4))

Report R_010:  255 rows  |  2025-01-01 → 2025-09-12
Imputed zero days: 1

Last 4 rows (boundary and any imputed zeros):


,report_id,date,daily_views,is_observed_day,is_imputed_zero
4346,R_010,2025-09-09,48,True,False
4347,R_010,2025-09-10,43,True,False
4348,R_010,2025-09-11,34,True,False
4349,R_010,2025-09-12,0,False,True


## 4. Validate the Daily Series

`validate_report_daily_series` runs 13 structural and logical checks:
required columns, nulls, non-negative values, integer dtype, grain uniqueness,
date continuity, indicator consistency, and row count.
It raises `SeriesValidationError` on any failure. A clean return proves all
checks passed.

In [6]:
try:
    validate_report_daily_series(mart_report_daily_series)
    print("✓  All 13 validation checks passed.")
except SeriesValidationError as e:
    print(f"VALIDATION FAILED:\n{e}")

_s = mart_report_daily_series
display(pd.DataFrame([
    {"check": "duplicate (report_id, date) pairs",
     "count": int(_s.duplicated(["report_id", "date"]).sum())},
    {"check": "null values in key columns",
     "count": int(_s[["report_id", "date", "daily_views"]].isna().sum().sum())},
    {"check": "imputed zero days",
     "count": int(_s["is_imputed_zero"].sum())},
    {"check": "reports with date gaps (non-contiguous)",
     "count": sum(
         1 for _, g in _s.groupby("report_id")
         if g["date"].sort_values().diff().dropna().ne(pd.Timedelta("1D")).any()
     )},
]))

✓  All 13 validation checks passed.


,check,count
0,"duplicate (report_id, date) pairs",0
1,null values in key columns,0
2,imputed zero days,2329
3,reports with date gaps (non-contiguous),0


## 5. Adoption Mart with Leakage-Safe Time-Series Features

`build_report_daily_adoption` aggregates raw events to the daily grain
(views, unique viewers, views-per-user). The result is left-joined onto
the canonical spine so imputed-zero days retain `unique_viewers = 0`.

`add_time_series_usage_features` then adds rolling and lag features.

**Leakage guard:** every rolling window applies `shift(1)` before
accumulating, so the window at day *t* covers [*t*-1, *t*-2, …] and
never includes `daily_views` from day *t* itself. The first row per
report is `NaN` — there is no prior-day history.

| Column | Role | Window |
|---|---|---|
| `views_7d` | historical predictor | sum over 7 days ending *t*-1 |
| `views_28d` | historical predictor | sum over 28 days ending *t*-1 |
| `viewers_7d` | historical predictor | sum over 7 days ending *t*-1 |
| `viewers_28d` | historical predictor | sum over 28 days ending *t*-1 |
| `wow_change_views` | historical predictor | (views_*t-1* − views_*t-8*) / views_*t-8* |

In [7]:
# Step 1: aggregate events → daily base mart
_base = build_report_daily_adoption(
    fact_report_views=fact_report_views,
    date_col="date_key",
    report_col="report_id",
    user_col="user_key",
    views_col="view_count",
)

# Step 2: join viewer counts onto spine; preserves imputed-zero rows
mart_report_daily_adoption = mart_report_daily_series.merge(
    _base[["date", "report_id", "unique_viewers", "views_per_user"]],
    on=["date", "report_id"],
    how="left",
)
mart_report_daily_adoption["unique_viewers"] = (
    mart_report_daily_adoption["unique_viewers"].fillna(0).astype(int)
)
mart_report_daily_adoption["views_per_user"] = (
    mart_report_daily_adoption["daily_views"]
    .div(mart_report_daily_adoption["unique_viewers"].replace(0, pd.NA))
    .fillna(0.0)
)

# Step 3: leakage-safe rolling features
mart_report_daily_adoption_ts = add_time_series_usage_features(mart_report_daily_adoption)

_new_cols = [c for c in mart_report_daily_adoption_ts.columns
             if c not in mart_report_daily_adoption.columns]
print(f"Adoption mart shape:          {mart_report_daily_adoption.shape}")
print(f"After TS features shape:      {mart_report_daily_adoption_ts.shape}")
print(f"Added TS columns:             {_new_cols}")

Adoption mart shape:          (11847, 7)
After TS features shape:      (11847, 12)
Added TS columns:             ['views_7d', 'views_28d', 'viewers_7d', 'viewers_28d', 'wow_change_views']


In [8]:
# Leakage check: first row per report must have NaN rolling values
_rolling_cols = ["views_7d", "views_28d", "viewers_7d", "viewers_28d"]
_first = mart_report_daily_adoption_ts.groupby("report_id").first()
print("First-row NaN (True = leakage guard working):")
print(_first[_rolling_cols].isna().all())

# Show 5 rows from week 2 onwards where rolling values are populated
display(
    mart_report_daily_adoption_ts[
        mart_report_daily_adoption_ts["report_id"] == mart_report_daily_adoption_ts["report_id"].iloc[0]
    ][["date", "daily_views", "views_7d", "views_28d", "wow_change_views"]]
    .iloc[7:12]
)

First-row NaN (True = leakage guard working):
views_7d       False
views_28d      False
viewers_7d     False
viewers_28d    False
dtype: bool


,date,daily_views,views_7d,views_28d,wow_change_views
7,2025-01-08,40,234.0,234.0,NaN
8,2025-01-09,31,235.0,274.0,0.025641
9,2025-01-10,44,223.0,305.0,-0.279070
10,2025-01-11,9,232.0,349.0,0.257143
11,2025-01-12,10,231.0,358.0,-0.100000


## 6. Report-Level Trend Features

`build_report_features` aggregates the daily series to **one row per report**
and computes window-based trend metrics.

| Metric | Definition | Minimum history | Denominator edge case |
|---|---|---|---|
| `usage_change_28d_pct` | (recent 28d − prior 28d) / prior 28d | 56 days | prior=0, recent>0 → null + `newly_active_flag` |
| `usage_trend_12w_slope` | OLS slope (views/week) over last 12 complete ISO weeks | 8 complete weeks | — |
| `newly_active_flag` | True when prior 28d = 0 and recent 28d > 0 | — | Undefined pct |
| `trend_history_sufficient` | True when series ≥ 56 calendar days | — | — |

**Role:** All are **diagnostic** and **not passed to SARIMA**.

In [9]:
mart_report_features = build_report_features(
    daily_adoption=mart_report_daily_adoption,
    fact_report_views=fact_report_views,
    report_performance=None,      # joined after performance mart is built
    dim_report=dim_report,
    dim_date=dim_date if not dim_date.empty else None,
)

print(f"Shape: {mart_report_features.shape}")
display(
    mart_report_features[
        ["report_id", "report_name",
         "recent_28d_views", "previous_28d_views",
         "usage_change_28d_pct", "newly_active_flag",
         "trend_history_sufficient", "usage_trend_12w_slope"]
    ].head(8)
)

Shape: (30, 16)


,report_id,report_name,recent_28d_views,previous_28d_views,usage_change_28d_pct,newly_active_flag,trend_history_sufficient,usage_trend_12w_slope
0,R_001,Commercial Finance Exposure Dashboard,1007,817,0.232558,False,True,2.024476
1,R_002,FX & Rates Risk Monitor,664,534,0.243446,False,True,1.965035
2,R_003,Headcount & Payroll Summary,565,707,-0.200849,False,True,-5.895105
3,R_004,New Business Volume Tracker,985,983,0.002035,False,True,0.255245
4,R_005,Digital Channel Adoption Report,754,803,-0.061021,False,True,0.531469
5,R_006,ESG & Sustainability Metrics Dashboard,688,594,0.158249,False,True,0.237762
6,R_007,Legacy Cost Allocation Report,236,317,-0.255521,False,True,-3.989510
7,R_008,Premises & Facilities Utilisation,220,245,-0.102041,False,True,-1.702797


In [10]:
# Denominator-handling check: newly_active reports must have null usage_change_28d_pct
_newly = mart_report_features[mart_report_features["newly_active_flag"] == True]
print(f"Newly-active reports (prior 28d = 0, recent > 0): {len(_newly)}")
if not _newly.empty:
    assert _newly["usage_change_28d_pct"].isna().all(), \
        "FAIL: newly_active reports should have null usage_change_28d_pct"
    print("✓  usage_change_28d_pct is null for all newly-active reports.")

_slope_n = mart_report_features["usage_trend_12w_slope"].notna().sum()
print(f"\nReports with trend slope available: {_slope_n} / {len(mart_report_features)}")
print(mart_report_features["usage_trend_12w_slope"].describe().round(2))

Newly-active reports (prior 28d = 0, recent > 0): 0

Reports with trend slope available: 30 / 30
count    30.00
mean     -0.12
std       2.56
min      -5.90
25%      -1.36
50%       0.10
75%       1.29
max       5.33
Name: usage_trend_12w_slope, dtype: float64


## 7. Engagement Context

Engagement features explain **how** users interact with a report at the
`date × report_id` grain. Two concentration metrics are kept distinct:

| Column | Calculation | Window | Risk signal |
|---|---|---|---|
| `top_1_user_view_share` | max(user_views) / total_views on this day | same-day actuals | Single-user dependency |
| `top_10pct_user_share` | top-10%-of-users views / total_views on this day | same-day actuals | Breadth vs concentration |
| `repeat_user_rate` | fraction of users seen on a prior day | same-day actuals | Recurring habit |
| `days_since_last_use` | calendar days since last observed event | same-day actuals | Idle risk |
| `avg_pages_per_user` | pages viewed per distinct user | same-day actuals | Depth of use |

**Role:** All are **diagnostic-only**. They reflect same-day actuals that cannot
be known at forecast time, so they are not valid SARIMA predictors.

In [11]:
mart_user_engagement = build_user_engagement_features(
    fact_report_views=fact_report_views,
    fact_page_views=fact_page_views,
    date_col="date_key",
    report_col="report_id",
    user_col="user_key",
)

print(f"Shape:   {mart_user_engagement.shape}")
print(f"Columns: {mart_user_engagement.columns.tolist()}")
display(mart_user_engagement.head(5))

Shape:   (10212, 7)
Columns: ['date', 'report_id', 'repeat_user_rate', 'top_1_user_view_share', 'top_10pct_user_share', 'days_since_last_use', 'avg_pages_per_user']


,date,report_id,repeat_user_rate,top_1_user_view_share,top_10pct_user_share,days_since_last_use,avg_pages_per_user
0,2025-01-01,R_001,0.000000,0.307692,0.358974,0,1.928571
1,2025-01-02,R_001,0.240000,0.441860,0.488372,1,1.640000
2,2025-01-03,R_001,0.458333,0.342857,0.400000,1,2.166667
3,2025-01-04,R_001,0.714286,0.400000,0.400000,1,2.428571
4,2025-01-05,R_001,0.833333,0.416667,0.416667,1,2.666667


In [12]:
# Confirm both concentration columns are present and the legacy name is absent
assert "top_1_user_view_share" in mart_user_engagement.columns, \
    "top_1_user_view_share missing from engagement mart"
assert "top_10pct_user_share" in mart_user_engagement.columns, \
    "top_10pct_user_share missing from engagement mart"
assert "top_user_share" not in mart_user_engagement.columns, \
    "Legacy ambiguous column top_user_share must not be present"
print("✓  Both concentration columns present; legacy name absent.")

# top_10pct >= top_1 when there are enough users (top-10% includes the top-1 user)
_higher = (mart_user_engagement["top_10pct_user_share"]
           >= mart_user_engagement["top_1_user_view_share"]).mean()
print(f"top_10pct >= top_1 in {_higher:.1%} of rows (expected near 100%)")

print("\nConcentration summary:")
display(
    mart_user_engagement[["top_1_user_view_share", "top_10pct_user_share"]]
    .describe().round(3)
)

✓  Both concentration columns present; legacy name absent.
top_10pct >= top_1 in 100.0% of rows (expected near 100%)

Concentration summary:


,top_1_user_view_share,top_10pct_user_share
count,10212.000,10212.000
mean,0.416,0.472
std,0.253,0.221
min,0.080,0.143
25%,0.238,0.333
50%,0.333,0.395
75%,0.500,0.500
max,1.000,1.000


## 8. Performance Context

`build_report_performance_features` aggregates load-time telemetry to the
`date × report_id` grain.

| Column | Definition |
|---|---|
| `avg_load_time` | Mean load time (ms) for this report-day |
| `p90_load_time` | 90th-percentile load time (ms) |
| `avg_load_time_7d` | 7-day rolling average of `avg_load_time` |
| `load_time_wow_change` | WoW fractional change in `avg_load_time` |
| `load_events` | Number of load events on this report-day |

**Role:** All are **diagnostic-only** in the current pipeline. `avg_load_time`
on day *t* is an actuals-based measure that cannot be known before day *t*
ends.

In [13]:
mart_report_performance = build_report_performance_features(
    fact_report_loads=fact_report_loads,
    date_col="date_key",
    report_col="report_id",
    load_time_col="load_time_ms",
)

print(f"Shape:   {mart_report_performance.shape}")
print(f"Columns: {mart_report_performance.columns.tolist()}")
display(mart_report_performance.head(5))

Shape:   (10212, 7)
Columns: ['date', 'report_id', 'avg_load_time', 'p90_load_time', 'avg_load_time_7d', 'load_time_wow_change', 'load_events']


,date,report_id,avg_load_time,p90_load_time,avg_load_time_7d,load_time_wow_change,load_events
0,2025-01-01,R_001,3509.428571,4045.9,3509.428571,NaN,28
1,2025-01-02,R_001,3637.440000,4317.6,3573.434286,NaN,25
2,2025-01-03,R_001,3402.625000,3768.7,3516.497857,NaN,24
3,2025-01-04,R_001,3253.714286,3770.8,3450.801964,NaN,7
4,2025-01-05,R_001,3391.000000,3604.0,3438.841571,NaN,6


In [14]:
_multi = mart_report_performance[mart_report_performance["load_events"] > 1]
print("p90 >= avg when >1 load event:",
      _multi["p90_load_time"].ge(_multi["avg_load_time"]).all())
print("No infinite load_time_wow_change:",
      not mart_report_performance["load_time_wow_change"]
          .isin([float("inf"), float("-inf")]).any())
_wow_null_rate = mart_report_performance["load_time_wow_change"].isna().mean()
print(f"load_time_wow_change null rate: {_wow_null_rate:.1%} "
      f"(expected for first ~7 rows per report)")
print("\nLoad-time summary (ms):")
display(mart_report_performance[["avg_load_time", "p90_load_time"]].describe().round(0))

p90 >= avg when >1 load event: True
No infinite load_time_wow_change: True
load_time_wow_change null rate: 2.1% (expected for first ~7 rows per report)

Load-time summary (ms):


,avg_load_time,p90_load_time
count,10212.0,10212.0
mean,3648.0,4082.0
std,1212.0,1204.0
min,500.0,500.0
25%,2980.0,3349.0
50%,3551.0,4028.0
75%,4167.0,4609.0
max,7752.0,8165.0


## 9. Assemble Outputs

Two marts are assembled:

1. **`mart_report_daily_series`** — already built in §3; no changes needed.
2. **`mart_report_daily_context`** — wide diagnostic table joining adoption
   + engagement + performance via `build_report_daily_context`.

A third table, `mart_report_insight_context`, is built by
`build_report_insight_context` after the forecasting pipeline runs
(in `run_report_analytics_pipeline.py`). It is not produced here because
forecast outputs are not yet available at feature-engineering time.

In [15]:
mart_report_daily_context = build_report_daily_context(
    mart_report_daily_adoption=mart_report_daily_adoption_ts,
    mart_user_engagement=mart_user_engagement,
    mart_report_performance=mart_report_performance,
)

print("mart_report_daily_series:")
print(f"  shape={mart_report_daily_series.shape}  "
      f"cols={mart_report_daily_series.columns.tolist()}")
print("\nmart_report_daily_context:")
print(f"  shape={mart_report_daily_context.shape}")
print(f"  cols={mart_report_daily_context.columns.tolist()}")

mart_report_daily_series:
  shape=(11847, 5)  cols=['report_id', 'date', 'daily_views', 'is_observed_day', 'is_imputed_zero']

mart_report_daily_context:
  shape=(11847, 22)
  cols=['report_id', 'date', 'daily_views', 'is_observed_day', 'is_imputed_zero', 'unique_viewers', 'views_per_user', 'views_7d', 'views_28d', 'viewers_7d', 'viewers_28d', 'wow_change_views', 'repeat_user_rate', 'top_1_user_view_share', 'top_10pct_user_share', 'days_since_last_use', 'avg_pages_per_user', 'avg_load_time', 'p90_load_time', 'avg_load_time_7d', 'load_time_wow_change', 'load_events']


## 10. Quality Summary

In [16]:
# ── Output shapes and date coverage ──────────────────────────────────────────
_coverage = []
for _name, _df in [
    ("mart_report_daily_series",  mart_report_daily_series),
    ("mart_report_daily_context", mart_report_daily_context),
]:
    _coverage.append({
        "mart": _name,
        "rows": len(_df),
        "cols": _df.shape[1],
        "distinct_reports": _df["report_id"].nunique(),
        "date_min": str(_df["date"].min().date()),
        "date_max": str(_df["date"].max().date()),
        "grain_unique": not _df.duplicated(["report_id", "date"]).any(),
    })
display(pd.DataFrame(_coverage))

,mart,rows,cols,distinct_reports,date_min,date_max,grain_unique
0,mart_report_daily_series,11847,5,30,2025-01-01,2026-03-31,True
1,mart_report_daily_context,11847,22,30,2025-01-01,2026-03-31,True


In [17]:
# ── Missing-value summary for mart_report_daily_context ──────────────────────
# Nulls in engagement/performance columns are expected for imputed-zero days
# (no events → no engagement mart row → left-join produces null).
_null_pct = (
    mart_report_daily_context.isna().mean().mul(100).round(1)
    .rename("null_%")
    .to_frame()
    .query("`null_%` > 0")
    .sort_values("null_%", ascending=False)
)
print("Columns with any nulls (context mart):")
print(f"Imputed-zero days: {mart_report_daily_context['is_imputed_zero'].sum():,} "
      f"— these rows have null engagement/performance values by construction.")
display(_null_pct)

Columns with any nulls (context mart):
Imputed-zero days: 2,329 — these rows have null engagement/performance values by construction.


,null_%
load_time_wow_change,21.4
wow_change_views,21.3
repeat_user_rate,19.7
top_1_user_view_share,19.7
top_10pct_user_share,19.7
days_since_last_use,19.7
avg_pages_per_user,19.7
avg_load_time,19.7
p90_load_time,19.7
avg_load_time_7d,19.7


In [18]:
# ── Feature-role summary from the registry ────────────────────────────────────
_reg = get_feature_registry()
print("Feature roles:")
display(
    _reg.groupby("feature_role")["feature_name"]
    .apply(lambda s: ", ".join(sorted(s)))
    .rename("features")
    .to_frame()
)
print(f"\nPredictor features (safe for SARIMA): {get_predictor_features()}")
print(f"Diagnostic features (context only):   "
      f"{get_diagnostic_features()[:6]} … ({len(get_diagnostic_features())} total)")

Feature roles:


,features
feature_role,
diagnostic-only,"avg_load_time, avg_load_time_7d, avg_pages_per..."
historical predictor,"viewers_28d, viewers_7d, views_28d, views_7d, ..."
identifier,"date, report_id"
known-in-advance predictor,"day_of_week, is_month_end, is_quarter_end, is_..."
target,daily_views



Predictor features (safe for SARIMA): ['views_7d', 'views_28d', 'viewers_7d', 'viewers_28d', 'wow_change_views', 'day_of_week', 'is_weekend', 'is_month_end', 'is_quarter_end']
Diagnostic features (context only):   ['unique_viewers', 'views_per_user', 'repeat_user_rate', 'top_1_user_view_share', 'top_10pct_user_share', 'days_since_last_use'] … (19 total)


In [19]:
# ── Schema checks ─────────────────────────────────────────────────────────────
_series_required = [
    "report_id", "date", "daily_views", "is_observed_day", "is_imputed_zero"
]
_context_required = [
    "report_id", "date", "daily_views",
    "views_7d", "views_28d", "wow_change_views",
    "repeat_user_rate",
    "top_1_user_view_share", "top_10pct_user_share",
    "avg_load_time", "p90_load_time",
]
for _label, _df, _req in [
    ("mart_report_daily_series",  mart_report_daily_series,  _series_required),
    ("mart_report_daily_context", mart_report_daily_context, _context_required),
]:
    _missing = [c for c in _req if c not in _df.columns]
    _status = "✓ PASS" if not _missing else f"✗ FAIL — missing: {_missing}"
    print(f"{_label}: {_status}")

mart_report_daily_series: ✓ PASS
mart_report_daily_context: ✓ PASS


## 11. Representative Examples

Four archetypes illustrate how the series and context features look under
different usage patterns. Reports are selected by archetype label from
`dim_report` so the selection is data-driven rather than hardcoded.

In [20]:
def _pick(substr: str):
    """Return the first report_id whose archetype contains substr."""
    mask = dim_report["archetype"].str.contains(substr, na=False)
    ids = dim_report.loc[mask, "report_id"]
    return ids.iloc[0] if not ids.empty else None

_SHOW_COLS = ["date", "report_id", "daily_views", "is_imputed_zero",
              "views_7d", "wow_change_views"]

def _show(label: str, rid, n: int = 6):
    if rid is None:
        print(f"[{label}] — no matching archetype found")
        return
    _df = mart_report_daily_context[mart_report_daily_context["report_id"] == rid]
    _arch = dim_report.loc[dim_report["report_id"] == rid, "archetype"].iloc[0]
    _zero_pct = _df["is_imputed_zero"].mean()
    print(f"── {label} ({rid}, archetype={_arch}) "
          f"| {len(_df)} rows | imputed-zero rate={_zero_pct:.1%} ──")
    display(_df[[c for c in _SHOW_COLS if c in _df.columns]].tail(n))

print("Available archetypes:", sorted(dim_report["archetype"].dropna().unique()))

Available archetypes: ['gradual_decline', 'high_volatility_exec', 'intermittent_specialist', 'launch_and_growth', 'month_end', 'quarter_end', 'replacement_cannibalized', 'stable_weekly', 'sudden_retirement', 'upward_adoption']


In [21]:
# Stable report: consistent weekly pattern, low wow_change_views variance
_show("Stable", _pick("stable"))

── Stable (R_001, archetype=stable_weekly) | 455 rows | imputed-zero rate=5.1% ──


,date,report_id,daily_views,is_imputed_zero,views_7d,wow_change_views
449,2026-03-26,R_001,49,False,287.0,1.464286
450,2026-03-27,R_001,48,False,284.0,-0.057692
451,2026-03-28,R_001,11,False,285.0,0.021277
452,2026-03-29,R_001,13,False,286.0,0.100000
453,2026-03-30,R_001,50,False,290.0,0.444444
454,2026-03-31,R_001,56,False,288.0,-0.038462


In [22]:
# Declining report: persistent negative wow_change_views
_show("Declining", _pick("decline"))

── Declining (R_007, archetype=gradual_decline) | 455 rows | imputed-zero rate=3.7% ──


,date,report_id,daily_views,is_imputed_zero,views_7d,wow_change_views
3179,2026-03-26,R_007,13,False,48.0,0.200000
3180,2026-03-27,R_007,13,False,52.0,0.444444
3181,2026-03-28,R_007,2,False,56.0,0.444444
3182,2026-03-29,R_007,2,False,56.0,0.000000
3183,2026-03-30,R_007,9,False,55.0,-0.333333
3184,2026-03-31,R_007,13,False,51.0,-0.307692


In [23]:
# Sparse report: high imputed-zero rate, intermittent usage
_sparse_id = _pick("sparse")
if _sparse_id is None:
    # Fallback: pick the report with the highest imputed-zero rate
    _zero_rates = (
        mart_report_daily_context.groupby("report_id")["is_imputed_zero"].mean()
    )
    _sparse_id = _zero_rates.idxmax()
_show("Sparse", _sparse_id)

── Sparse (R_014, archetype=intermittent_specialist) | 455 rows | imputed-zero rate=92.5% ──


,date,report_id,daily_views,is_imputed_zero,views_7d,wow_change_views
5767,2026-03-26,R_014,0,True,1.0,NaN
5768,2026-03-27,R_014,0,True,0.0,-1.0
5769,2026-03-28,R_014,0,True,0.0,NaN
5770,2026-03-29,R_014,0,True,0.0,NaN
5771,2026-03-30,R_014,0,True,0.0,NaN
5772,2026-03-31,R_014,0,True,0.0,NaN


In [24]:
# Retired report: series ends at retire_date; boundary day is an imputed zero
_retired_id = dim_report[dim_report["retire_date"].notna()]["report_id"].iloc[0]
_show("Retired (zero-view boundary)", _retired_id, n=4)

── Retired (zero-view boundary) (R_010, archetype=sudden_retirement) | 255 rows | imputed-zero rate=0.4% ──


,date,report_id,daily_views,is_imputed_zero,views_7d,wow_change_views
4346,2025-09-09,R_010,48,False,225.0,0.236842
4347,2025-09-10,R_010,43,False,227.0,0.043478
4348,2025-09-11,R_010,34,False,244.0,0.653846
4349,2025-09-12,R_010,0,True,243.0,-0.028571


## 12. Limitations

| Limitation | Impact | Mitigation path |
|---|---|---|
| **SARIMA is univariate** | Cannot capture explicit day-of-week, holiday, or event-calendar effects | Add known-in-advance calendar flags as SARIMAX regressors |
| **Engagement features are same-day actuals** | Cannot be forecast inputs today | Diagnostic use only; label `available_at_forecast_time=False` in registry |
| **Performance features are same-day actuals** | Same constraint | Diagnostic use only; forecast or lag before using as regressor |
| **`usage_trend_12w_slope` requires ≥ 8 complete weeks** | Null for recently launched reports | `trend_history_sufficient` flags when to suppress |
| **`usage_change_28d_pct` requires ≥ 56 days** | Null for recently launched reports | `newly_active_flag` distinguishes undefined-denominator case |
| **Imputed zeros on weekends suppress rolling means** | Views-per-week appears lower for weekend-heavy series | Downstream models can weight `is_observed_day` rows differently |
| **Active-period anchoring depends on `dim_report` quality** | Incorrect launch/retire dates cause boundary errors | `validate_report_daily_series` catches date-gap violations |

## 13. Save Outputs

Each mart is saved to its standard path in `data/processed/`. No
notebook-specific copies with alternative column names are created.

In [25]:
mart_report_daily_series.to_csv(SERIES_OUT, index=False)
mart_report_daily_adoption.to_csv(ADOPTION_OUT, index=False)
mart_report_daily_adoption_ts.to_csv(TS_OUT, index=False)
mart_user_engagement.to_csv(ENGAGEMENT_OUT, index=False)
mart_report_performance.to_csv(PERFORMANCE_OUT, index=False)
mart_report_daily_context.to_csv(CONTEXT_OUT, index=False)

for _name, _path in [
    ("mart_report_daily_series",     SERIES_OUT),
    ("mart_report_daily_adoption",   ADOPTION_OUT),
    ("mart_report_daily_adoption_ts", TS_OUT),
    ("mart_user_engagement",          ENGAGEMENT_OUT),
    ("mart_report_performance",       PERFORMANCE_OUT),
    ("mart_report_daily_context",     CONTEXT_OUT),
]:
    _kb = _path.stat().st_size / 1024
    print(f"{_name:<40}  → {_path.name}  ({_kb:.0f} KB)")

mart_report_daily_series                  → mart_report_daily_series.csv  (353 KB)
mart_report_daily_adoption                → mart_report_daily_adoption.csv  (493 KB)
mart_report_daily_adoption_ts             → mart_report_daily_adoption_ts_features.csv  (870 KB)
mart_user_engagement                      → mart_user_engagement.csv  (611 KB)
mart_report_performance                   → mart_report_performance.csv  (782 KB)
mart_report_daily_context                 → mart_report_daily_context.csv  (1915 KB)
